In [1]:
import torch
import random
import torch.nn as nn


In [2]:
class SimpleEnviroment:
    def __init__(self):
        self.states = ["s1", "s2"]
        self.actions= ["a1", "a2", "a3"]

        self.trans = {
            ("s1", "a1"): {("s1", +1): 1 },
            ("s1", "a2"): {("s1", +4): 0.7, ("s2", +5) : 0.3 },
            ("s2", "a1"): {("s2", +1): 1 },
            ("s2", "a2"): {("s1", -3): 0.8, ("s2", +2) : 0.2 },
            ("s2", "a3"): {("s2", 0): 1 }
        }

    def reset(self):
        self.current_state = random.choice(self.states)
        return self.current_state 
    
    def step(self, action):
        trans_value = self.trans[(self.current_state, action)]
        next_state, reward = random.choices(list(trans_value.keys()), weights=list(trans_value.values()))[0]
        self.current_state = next_state
        return next_state, reward
    


In [3]:

class GridEnvironment:
    def __init__(self, size):
        self.size = size
        # self.terminate_state = [(0,0), (size-1, size-1)]
        self.terminate_state = [(size-1, size-1)]
        self.states = [(i,j) for i in range(self.size) for j in range(self.size)
                       if (i,j) not in self.terminate_state]
        self.actions = [(1,0), (0,1), (-1,0), (0,-1)]

    def reset(self): 
        self.current_state = random.choice(self.states)
        return self.current_state
    
    def step(self, action):
        next_state = (self.current_state[0] + action[0], self.current_state[1] + action[1])
        # Check if next_state is valid (within grid boundaries)
        if not (0 <= next_state[0] < self.size and 0 <= next_state[1] < self.size):
            # If out of bounds, stay in the current state
            next_state = self.current_state
        reward = -1 
        self.current_state = next_state
        return next_state, reward

def deterministic_policy(env, state):
    valid_actions = []
    for action in env.actions:
        next_state = (state[0] + action[0], state[1] + action[1])
        if 0 <= next_state[0] < env.size and 0 <= next_state[1] < env.size:
            valid_actions.append(action)
    
    if valid_actions:
        return random.choice(valid_actions)
    return (0, 0)  # Default action if no valid actions (shouldn't happen with boundary checking)

def run_episode(env, max_steps=50):
    state = env.reset()
    total_sarsa = []
    
    for _ in range(max_steps):
        action = deterministic_policy(env, state)
        next_state, reward = env.step(action)
        
        if next_state in env.terminate_state:
            print("Reached terminal state:", next_state)
            total_sarsa.extend([state, action, reward, next_state, "end"])
            break
        else:
            total_sarsa.extend([state, action, reward])
        
        state = next_state
    
    return total_sarsa

In [4]:
def run_episode_update_q(env, max_steps=100):
    state = env.reset()
    total_sarsa = []
    
    for _ in range(max_steps):
        action = deterministic_policy(env, state)
        next_state, reward = env.step(action)
        
        if next_state in env.terminate_state:
            print("Reached terminal state:", next_state)
            total_sarsa.extend([state, action, reward, next_state, 100])
            break
        else:
            total_sarsa.extend([state, action, reward])
        
        state = next_state
    
    return total_sarsa

In [5]:
class QFunction(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(QFunction, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x):
        return self.network(x)

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
qfunction=QFunction(4, 30, 1)

In [7]:
qfunction = qfunction.to(device)

In [8]:
device_test = next(qfunction.parameters()).device

In [9]:
device_test

device(type='cuda', index=0)

In [10]:
gamma= 0.5
theta = 1e-10


In [11]:
env=GridEnvironment(5)



In [12]:
state = env.reset()
episodes = []
for i in range(0, 50):
    episode=run_episode_update_q(env, 1000)
    episodes.append(episode)

Reached terminal state: (4, 4)
Reached terminal state: (4, 4)
Reached terminal state: (4, 4)
Reached terminal state: (4, 4)
Reached terminal state: (4, 4)
Reached terminal state: (4, 4)
Reached terminal state: (4, 4)
Reached terminal state: (4, 4)
Reached terminal state: (4, 4)
Reached terminal state: (4, 4)
Reached terminal state: (4, 4)
Reached terminal state: (4, 4)
Reached terminal state: (4, 4)
Reached terminal state: (4, 4)
Reached terminal state: (4, 4)
Reached terminal state: (4, 4)
Reached terminal state: (4, 4)
Reached terminal state: (4, 4)
Reached terminal state: (4, 4)
Reached terminal state: (4, 4)
Reached terminal state: (4, 4)
Reached terminal state: (4, 4)
Reached terminal state: (4, 4)
Reached terminal state: (4, 4)
Reached terminal state: (4, 4)
Reached terminal state: (4, 4)
Reached terminal state: (4, 4)
Reached terminal state: (4, 4)
Reached terminal state: (4, 4)
Reached terminal state: (4, 4)
Reached terminal state: (4, 4)
Reached terminal state: (4, 4)
Reached 

In [13]:
# Basic optimizer setup
qfunction = QFunction(4, 30, 1)  # Input: state (x,y), Output: Q-value

# optimizer = torch.optim.Adam(qfunction.parameters(), lr=0.001)

# With more options
optimizer = torch.optim.Adam(
    qfunction.parameters(),
    lr=0.001,           # Learning rate
    betas=(0.9, 0.999), # Exponential decay rates for moment estimates
    eps=1e-8,           # Term added for numerical stability
    weight_decay=0      # L2 penalty (regularization)
)

# Alternative optimizers
# sgd_optimizer = torch.optim.SGD(qfunction.parameters(), lr=0.01, momentum=0.9)
# rmsprop_optimizer = torch.optim.RMSprop(qfunction.parameters(), lr=0.01)


In [14]:
def update_q(episodes, qfunction, optimizer, gamma=0.5):
    device = next(qfunction.parameters()).device
    for k in range(0,30):
        for q in range(0, len(episodes)):
            for i in range(0, len(episodes[q])-5, 3):
                state = episodes[q][i]
                action = episodes[q][i+1]
                reward = episodes[q][i+2]
                next_state = episodes[q][i+3]
                next_action = episodes[q][i+4]
                state_tensor = torch.FloatTensor(state+action).to(device)
                next_state_tensor = torch.FloatTensor(next_state+next_action).to(device)
                output_state=qfunction(state_tensor)
                with torch.no_grad():  # Don't compute gradients for target network
                        output_next_state = qfunction(next_state_tensor)
                        target = torch.tensor([[reward + gamma * output_next_state.item()]])
                
                # Compute TD error (difference between current Q and target)
                td_error = target - output_state
                
                # Use MSE loss (squared TD error)
                loss = td_error.pow(2).mean()
                
                # Update network
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            state = episodes[q][-5]
            action = episodes[q][-4]
            reward = episodes[q][-1]
            try:
                state_tensor = torch.FloatTensor(list(state+action)).to(device)
                output_state=qfunction(state_tensor).to(device)
                target = torch.tensor([[reward]], dtype=torch.float32).to(device)

                td_error = target - output_state
                loss = td_error.pow(2).mean()
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            except Exception as e :
                print(e)
                print(state)
                print(action)
    return qfunction
            
    

In [15]:
final_qfucntion=update_q(episodes,qfunction,optimizer )

In [19]:
final_qfucntion =final_qfucntion.to(device)

In [20]:
device_test = next(final_qfucntion.parameters()).device

In [21]:
device_test

device(type='cuda', index=0)

In [22]:
q = 1
for i in range(0, len(episodes[q])-5, 3):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    state = episodes[q][i]
    action = episodes[q][i+1]
    reward = episodes[q][i+2]
    next_state = episodes[q][i+3]
    next_action = episodes[q][i+4]
    state_tensor = torch.FloatTensor(state+action).to(device)
    next_state_tensor = torch.FloatTensor(next_state+next_action)
    output_state=final_qfucntion(state_tensor)



(0, 4)
tensor([-1.1819], device='cuda:0', grad_fn=<ViewBackward0>)
-------------------------
(0, 3)
tensor([0.6841], device='cuda:0', grad_fn=<ViewBackward0>)
-------------------------
(0, 4)
tensor([-1.1819], device='cuda:0', grad_fn=<ViewBackward0>)
-------------------------
(0, 3)
tensor([1.4377], device='cuda:0', grad_fn=<ViewBackward0>)
-------------------------
(1, 3)
tensor([-1.3635], device='cuda:0', grad_fn=<ViewBackward0>)
-------------------------
(1, 2)
tensor([-0.3480], device='cuda:0', grad_fn=<ViewBackward0>)
-------------------------
(2, 2)
tensor([-1.8773], device='cuda:0', grad_fn=<ViewBackward0>)
-------------------------
(2, 1)
tensor([-1.8301], device='cuda:0', grad_fn=<ViewBackward0>)
-------------------------
(1, 1)
tensor([-1.4277], device='cuda:0', grad_fn=<ViewBackward0>)
-------------------------
(0, 1)
tensor([-0.7268], device='cuda:0', grad_fn=<ViewBackward0>)
-------------------------
(1, 1)
tensor([-1.4277], device='cuda:0', grad_fn=<ViewBackward0>)
-----

In [39]:
final=(1,1) + (0, -1)
state_tensor = torch.FloatTensor(final).to(device)
output_state=final_qfucntion(state_tensor)
print(output_state)

tensor([-1.9549], device='cuda:0', grad_fn=<ViewBackward0>)
